In [1]:
# ============================================================
# Model 6: IndoBERT Baseline
# Terminal Table Output Only
# ============================================================

import os
import re
import random
import numpy as np
import pandas as pd
import torch

from collections import Counter
from torch.utils.data import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

# ============================================================
# CONFIG
# ============================================================

DATA_DIR = "/net/pr2/projects/plgrid/plggman01/HateSpeech"

PATH_IDHSD = os.path.join(DATA_DIR, "IDHSD_RIO_unbalanced_713_2017.txt")
PATH_572 = os.path.join(DATA_DIR, "572-hate-speech-dataset.csv")
PATH_RE = os.path.join(DATA_DIR, "re_dataset.csv")
PATH_KAMUSALAY = os.path.join(DATA_DIR, "new_kamusalay.csv")

MODEL_NAME = "xlm-roberta-base"
MODEL_LABEL = "XLM-R"

SEED = 42
LABEL_FRACTIONS = [0.05, 0.10, 0.20]

MAX_LEN = 128
BATCH_SIZE = 8
EPOCHS = 5

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ============================================================
# DATA LOADING
# ============================================================

def safe_read_csv(path, sep=",", header="infer"):
    for enc in ["utf-8", "utf-8-sig", "latin1", "cp1252"]:
        try:
            return pd.read_csv(path, sep=sep, encoding=enc, header=header)
        except Exception:
            pass
    raise ValueError(f"Cannot read {path}")


def normalize_binary_label(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, str):
        v = value.strip().lower()
        if v in {"hs", "hate", "hate_speech", "1", "true", "yes"}:
            return 1
        if v in {"non_hs", "non-hs", "non hs", "0", "false", "no"}:
            return 0

    if isinstance(value, (int, float, np.integer, np.floating)):
        return int(value)

    return np.nan


URL_PATTERN = re.compile(r"http\S+|www\S+|https\S+")
MENTION_PATTERN = re.compile(r"@\w+")
HASHTAG_PATTERN = re.compile(r"#(\w+)")
NON_ALNUM_PATTERN = re.compile(r"[^a-zA-Z0-9\s!?]")
MULTISPACE_PATTERN = re.compile(r"\s+")
REPEAT_CHAR_PATTERN = re.compile(r"(.)\1{2,}")

LAUGHTER_VARIANTS = {
    "wkwk": "tertawa",
    "wkwkwk": "tertawa",
    "haha": "tertawa",
    "hahaha": "tertawa",
    "hehe": "tertawa"
}


def basic_clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = text.replace("\\n", " ")
    text = text.replace("\\/", "/")
    text = REPEAT_CHAR_PATTERN.sub(r"\1\1", text)
    text = URL_PATTERN.sub(" ", text)
    text = MENTION_PATTERN.sub(" ", text)
    text = HASHTAG_PATTERN.sub(lambda m: f" {m.group(1)} ", text)
    text = text.lower()
    text = NON_ALNUM_PATTERN.sub(" ", text)
    text = MULTISPACE_PATTERN.sub(" ", text).strip()

    return text


def load_kamusalay_dict(path):
    df = safe_read_csv(path)
    c1, c2 = df.columns[:2]

    slang = {}
    for _, row in df.iterrows():
        src = str(row[c1]).strip().lower()
        tgt = str(row[c2]).strip().lower()

        if src and src != "nan":
            slang[src] = tgt

    return slang


def preprocess_text(text, slang_dict):
    text = basic_clean_text(text)

    tokens = []
    for tok in text.split():
        tok = LAUGHTER_VARIANTS.get(tok, tok)
        tok = slang_dict.get(tok, tok)
        tokens.append(tok)

    return " ".join(tokens)


def load_idhsd_dataset(path):
    df = safe_read_csv(path, sep="\t")
    df = df.rename(columns={"Label": "raw_label", "Tweet": "text"})
    df["label"] = df["raw_label"].apply(normalize_binary_label)
    return df[["text", "label"]].dropna()


def load_572_dataset(path):
    df = safe_read_csv(path)

    text_col = next((c for c in ["comment_text", "tweet", "Tweet", "text", "Text"] if c in df.columns), None)
    label_col = next((c for c in ["class", "label", "Label", "HS"] if c in df.columns), None)

    if text_col is None or label_col is None:
        raise ValueError(f"Cannot detect text/label columns in {path}")

    df = df.rename(columns={text_col: "text", label_col: "raw_label"})
    df["label"] = df["raw_label"].apply(normalize_binary_label)

    return df[["text", "label"]].dropna()


def load_re_dataset(path):
    df = safe_read_csv(path)

    text_col = next((c for c in ["Tweet", "tweet", "text", "Text"] if c in df.columns), None)

    if text_col is None:
        raise ValueError(f"Cannot detect text column in {path}")

    if "HS" in df.columns:
        label_col = "HS"
    elif "label" in df.columns:
        label_col = "label"
    else:
        raise ValueError(f"Cannot detect label column in {path}")

    df = df.rename(columns={text_col: "text", label_col: "raw_label"})
    df["label"] = df["raw_label"].apply(normalize_binary_label)

    return df[["text", "label"]].dropna()


def merge_datasets():
    slang_dict = load_kamusalay_dict(PATH_KAMUSALAY)

    data = pd.concat([
        load_idhsd_dataset(PATH_IDHSD),
        load_572_dataset(PATH_572),
        load_re_dataset(PATH_RE)
    ], ignore_index=True)

    data = data.drop_duplicates(subset=["text"]).reset_index(drop=True)
    data["clean_text"] = data["text"].apply(lambda x: preprocess_text(x, slang_dict))
    data = data[data["clean_text"].str.len() > 2].reset_index(drop=True)

    print("=" * 70)
    print("FINAL DATASET SIZE :", len(data))
    print("LABEL DISTRIBUTION :", Counter(data["label"]))
    print("DEVICE             :", DEVICE)
    print("MODEL              :", MODEL_NAME)
    print("=" * 70)

    return data

# ============================================================
# DATASET CLASS
# ============================================================

class HFDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=MAX_LEN
        )
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            k: torch.tensor(v[idx])
            for k, v in self.encodings.items()
        }
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# ============================================================
# METRICS
# ============================================================

def compute_metrics_hf(eval_pred):
    logits, labels = eval_pred

    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "precision": precision_score(labels, preds, average="macro", zero_division=0),
        "recall": recall_score(labels, preds, average="macro", zero_division=0),
        "roc_auc": roc_auc_score(labels, probs)
    }


def evaluate_predictions(y_true, y_pred, y_prob):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob)
    }


def find_best_threshold(y_true, y_prob):
    best_t = 0.5
    best_f1 = -1

    for t in np.arange(0.20, 0.81, 0.02):
        pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, pred, average="macro")

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    return best_t

# ============================================================
# TRAIN AND TEST INDOBERT
# ============================================================

def run_experiment(train_df, val_df, test_df, frac):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )

    train_ds = HFDataset(
        train_df["clean_text"].tolist(),
        train_df["label"].astype(int).tolist(),
        tokenizer
    )

    val_ds = HFDataset(
        val_df["clean_text"].tolist(),
        val_df["label"].astype(int).tolist(),
        tokenizer
    )

    test_ds = HFDataset(
        test_df["clean_text"].tolist(),
        test_df["label"].astype(int).tolist(),
        tokenizer
    )

    args = TrainingArguments(
        output_dir=f"./tmp_model7_mbert_{frac}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        logging_steps=50,
        save_total_limit=1,
        seed=SEED,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics_hf,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()

    val_logits = trainer.predict(val_ds).predictions
    val_prob = torch.softmax(torch.tensor(val_logits), dim=1).numpy()[:, 1]
    best_t = find_best_threshold(val_df["label"].astype(int).values, val_prob)

    test_logits = trainer.predict(test_ds).predictions
    test_prob = torch.softmax(torch.tensor(test_logits), dim=1).numpy()[:, 1]
    test_pred = (test_prob >= best_t).astype(int)

    result = evaluate_predictions(
        test_df["label"].astype(int).values,
        test_pred,
        test_prob
    )

    result["fraction"] = frac
    result["train_size"] = len(train_df)
    result["threshold"] = best_t

    return result

# ============================================================
# PRINT TABLE
# ============================================================

def print_results_table(results):
    print("\n" + "=" * 105)
    print("{:<10} {:<12} {:<12} {:<10} {:<10} {:<10} {:<10} {:<10}".format(
        "Fraction",
        "TrainSize",
        "Threshold",
        "Accuracy",
        "MacroF1",
        "Precision",
        "Recall",
        "ROC_AUC"
    ))
    print("=" * 105)

    for r in results:
        print("{:<10} {:<12} {:<12.4f} {:<10.4f} {:<10.4f} {:<10.4f} {:<10.4f} {:<10.4f}".format(
            r["fraction"],
            r["train_size"],
            r["threshold"],
            r["accuracy"],
            r["macro_f1"],
            r["precision"],
            r["recall"],
            r["roc_auc"]
        ))

    print("=" * 105)

# ============================================================
# MAIN
# ============================================================

data = merge_datasets()

train_val_df, test_df = train_test_split(
    data,
    test_size=0.20,
    random_state=SEED,
    stratify=data["label"]
)

train_df_full, val_df = train_test_split(
    train_val_df,
    test_size=0.10,
    random_state=SEED,
    stratify=train_val_df["label"]
)

results = []

for frac in LABEL_FRACTIONS:
    print("\n" + "=" * 80)
    print(f"Running Model 6: IndoBERT | Label Fraction = {frac}")
    print("=" * 80)

    labeled_train_df, _ = train_test_split(
        train_df_full,
        train_size=frac,
        random_state=SEED,
        stratify=train_df_full["label"]
    )

    res = run_experiment(
        labeled_train_df,
        val_df,
        test_df,
        frac
    )

    results.append(res)

print_results_table(results)

print("\nMODEL 8: XLM-R BASELINE FINISHED.")

2026-04-26 23:13:20.472465: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


FINAL DATASET SIZE : 13899
LABEL DISTRIBUTION : Counter({0.0: 7966, 1.0: 5933})
DEVICE             : cuda
MODEL              : xlm-roberta-base

Running Model 6: IndoBERT | Label Fraction = 0.05


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/net/tscratch/people/plgshoffan/shoffan01/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Precision,Recall,Roc Auc
1,No log,0.683880,0.573741,0.370356,0.586811,0.501588,0.761813
2,0.684700,0.509758,0.748201,0.745031,0.744028,0.747828,0.826436
3,0.684700,0.491990,0.770683,0.767149,0.766046,0.769057,0.846630
4,0.502700,0.476558,0.790468,0.784894,0.786530,0.783649,0.862289
5,0.404100,0.481137,0.791367,0.787707,0.786771,0.788985,0.864414


/net/tscratch/people/plgshoffan/shoffan01/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/net/tscratch/people/plgshoffan/shoffan01/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/net/tscratch/people/plgshoffan/shoffan01/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/net/tscratch/people/plgshoffan/shoffan01/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vecto

/net/tscratch/people/plgshoffan/shoffan01/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(



Fraction   TrainSize    Threshold    Accuracy   MacroF1    Precision  Recall     ROC_AUC   
0.05       500          0.4600       0.5759     0.3799     0.6043     0.5046     0.6925    
0.1        1000         0.4800       0.7478     0.7428     0.7424     0.7433     0.8246    
0.2        2001         0.4800       0.7960     0.7931     0.7918     0.7955     0.8745    

MODEL 8: XLM-R BASELINE FINISHED.


In [2]:
import os
os.getcwd()

'/net/tscratch/people/plgshoffan/HateSpeech/paper_outputs_hybrid_plus/Experiment Used'